In [203]:
	
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
sys.path.append("../")

from shared_utils.generate import format_conversation, transform_conversations
from early_exit.util import module_name_is_layer_base
import numpy as np

from shared_utils.data import CSVPromptDataset
from shared_utils.load import get_model, get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text

from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode

from shared_utils.load import get_model, get_tokenizer, configs_from_yaml
import random

import torch
from torch.optim import Adam
from torch.nn import functional as F
from torch.utils.data import DataLoader

import sys
sys.path.append("../")

from shared_utils.data import CSVPromptDataset
from shared_utils.load import get_model, get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text

from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode

import wandb
import pandas as pd
import numpy as np
import string
import html
import matplotlib.colors as mcolors

In [204]:
import torch.nn as nn
from collections import defaultdict
from functools import partial

class ActivationLens:
    """
    A utility class to hook multiple layers of a PyTorch model and collect their
    activations during a forward pass. It is designed for analyses like "Logic Lens,"
    where you want to inspect the intermediate representations of a model.

    The class can be used as a context manager to ensure hooks are automatically removed.

    Attributes:
        activations (defaultdict): A dictionary mapping layer_path (str) to a list
                                   of activation tensors from that layer.
    """

    def __init__(self):
        """Initializes the ActivationLens."""
        self.activations = defaultdict(list)
        self._hook_handles = []
        self._model = None

    def _create_hook_fn(self, layer_path: str):
        """
        Factory function to create a hook function for a specific layer.
        The created hook function knows its layer_path and stores the activation
        in the correct place in our `activations` dictionary.
        """
        def _hook_fn(module, input_tensors, output_tensor):
            # The output of some layers might be a tuple; we're often interested in the first element.
            activation = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
            self.activations[layer_path].append(activation.detach().cpu())
        return _hook_fn

    def register(self, model: nn.Module, layer_paths: list[str]):
        """
        Registers forward hooks to a list of specific layers within the model.

        Args:
            model (nn.Module): The model to hook.
            layer_paths (list[str]): A list of dot-separated string paths to the target layers.
        """
        self._model = model
        self.remove_hooks() # Clear any existing hooks before registering new ones

        for path in layer_paths:
            try:
                # Navigate to the target layer
                target_layer = model
                for part in path.split('.'):
                    target_layer = getattr(target_layer, part)

                # Register the hook and store the handle
                hook_fn = self._create_hook_fn(path)
                handle = target_layer.register_forward_hook(hook_fn)
                self._hook_handles.append(handle)
                print(f"✅ Hook registered on '{type(target_layer).__name__}' at: {path}")

            except AttributeError:
                print(f"⚠️ Error: Could not find layer at path: {path}. Skipping.")
    
    def remove_hooks(self):
        """Removes all registered hooks."""
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles = []

    def clear_activations(self):
        """Clears all collected activations, but leaves the hooks in place."""
        self.activations.clear()

    # --- Context Manager Methods for clean, automatic hook removal ---
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # When the `with` block is exited, automatically remove all hooks
        self.remove_hooks()
        print("\n✨ All hooks automatically removed.")

In [205]:
# LOAD IN EXPERIMENT ARGS
# num_epoch = 1                     # args.num_epoch
num_exit_samples = 1                  # args.num_exit_samples
device = "cpu"                    # args.device
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"                    # args.model_name
model_config_path = "../config_deepseek.yaml"                     # args.model_config_path
dataset_path = "../results_and_data/early_exit_sft_dataset/test/data.csv"                  # args.dataset_path
prompt_config_path = "../results_and_data/early_exit_sft_dataset/test/prompt_config.json"                    # args.prompt_config_path
batch_size = 1                    # args.batch_size -- might want to sort out batching, but increasing num_exit_samples might be better + less effort

# LOAD IN THE MODEL AND TOKENIZER
tokenizer = get_tokenizer(model_name)
config = configs_from_yaml(model_config_path, tokenizer.eos_token_id)
model = get_model(model_name, config['model'], device)


# LOAD IN DATASET
dataset = CSVPromptDataset(dataset_path, prompt_config_path)
dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=dataset.collate_fn, shuffle=True)


# ENABLE EARLY EXITING
model = replace_attention_layers(model, config['lora'], device)

replacing generate_layer_type_without_early_exit_decision_head layer model.layers.0
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.1
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.2
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.3
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.4
replacing layer model.layers.5
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.6
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.7
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.8
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.9
replacing layer model.layers.10
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.11
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.12
replacing g

In [206]:
set_transformer_early_exit_mode(model, 'sft_teacher')


In [232]:
import torch
import torch.nn as nn
from collections import defaultdict

# --- 1. Minimal Class to Collect Activations ---

class ActivationLens:
    """A minimal class to hook model layers and collect activations."""
    def __init__(self):
        self.activations = defaultdict(list)
        self._hook_handles = []

    def _create_hook_fn(self, layer_path: str):
        """Creates a hook function that saves the output of a specific layer."""
        def _hook_fn(module, input, output):
            # The actual activation tensor is often the first element of the output
            activation = output[0] if isinstance(output, tuple) else output
            self.activations[layer_path].append(activation.detach().cpu())
        return _hook_fn

    def register(self, model: nn.Module, layer_paths: list[str]):
        """Registers a forward hook on each layer in the list."""
        for path in layer_paths:
            try:
                target_layer = model
                for part in path.split('.'):
                    target_layer = getattr(target_layer, part)
                handle = target_layer.register_forward_hook(self._create_hook_fn(path))
                self._hook_handles.append(handle)
            except AttributeError:
                print(f"⚠️ Warning: Could not find layer at path: {path}. Skipping.")
    
    def remove_hooks(self):
        """Removes all registered hooks to clean up."""
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles = []

# --- 2. Main Script to Generate and Print Outputs ---

# NOTE: Make sure your `model`, `tokenizer`, `config`, `generate_text`, and `device`
# variables are already defined and loaded.

# Define all layers to inspect (0-27 plus the final normalization)
num_layers = 28
layer_paths_to_hook = [f'base_model.model.model.layers.{i}' for i in range(num_layers)]
layer_paths_to_hook.append('base_model.model.model.norm')

# Instantiate the lens and run the model once to collect all activations
lens = ActivationLens()
lens.register(model, layer_paths_to_hook)

prompt = "Mrs. Snyder used to spend 40% of her monthly income on rent and utilities. Her salary was recently increased by $600 so now her rent and utilities only amount to 25% of her monthly income. How much was her previous monthly income?"

system_prompt = "You are a helpful assistant."
prefiller = ""

print("\n--- Running Model to Collect Activations ---")
with torch.no_grad():
    # We only need the model to run; the activations are collected by the hooks
    decoded_response, _ = generate_text(
        model=model,
        prompt=prompt,
        system_prompt=system_prompt,
        prefiller=prefiller,
        tokenizer=tokenizer,
        generation_config=config['generation'],
        device=device
    )
print("--- Model Run Complete ---\n")

# --- 3. Process and Save Output from Selected Layers to DataFrame ---

print("="*40)
print("--- Processing Layers (mod 5 + Final) ---")
print("="*40 + "\n")

# Sort the layers numerically for a clean printout
sorted_layers = sorted(
    lens.activations.keys(),
    key=lambda x: int(x.split('.')[4]) if 'layers' in x else float('inf')
)

# Store results in a list for DataFrame
layer_outputs_data = []

for path in sorted_layers:
    layer_activations = lens.activations[path]
    if not layer_activations:
        continue

    # Get the layer number for filtering
    if 'layers' in path:
        layer_num = int(path.split('.')[4])
        layer_label = f"Layer {layer_num}"
        # Only process layers where layer_num % 5 == 0
        if layer_num % 5 != 0:
            continue
    else:
        # Skip Final Norm layer
        continue

    # Concatenate hidden states from all generation steps into one tensor
    full_sequence_hidden_states = torch.cat(layer_activations, dim=1).to(device)

    # Use the model's readout head to get token probabilities (logits)
    logits = model.early_exit_hidden_state_readout(full_sequence_hidden_states)
    
    # Find the most likely token ID for each position in the sequence
    predicted_token_ids = logits.argmax(-1)
    
    # Decode the sequence of token IDs into human-readable text
    text_from_layer = tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)

    # Store in list WITHOUT CLEANING (will clean in visualization)
    layer_outputs_data.append({
        'layer_number': layer_num,
        'layer_label': layer_label,
        'generated_text': text_from_layer
    })
    
    # Print the result for the current layer
    print(f"--- {layer_label} ---")
    print(f"{text_from_layer[:200]}...\n")  # Print first 200 chars

# Add the actual final output
layer_outputs_data.append({
    'layer_number': 'actual_output',
    'layer_label': 'Output',
    'generated_text': decoded_response
})

# --- 4. Create DataFrame and Save ---
layer_outputs_dataframe = pd.DataFrame(layer_outputs_data)

# Save to CSV in the same folder as the notebook
output_path = "layer_outputs_mod5.csv"
layer_outputs_dataframe.to_csv(output_path, index=False)
print(f"\n✅ DataFrame saved to: {output_path} (same folder as notebook)")

# Display the DataFrame
print("\n--- DataFrame Preview ---")
print(layer_outputs_dataframe[['layer_number', 'layer_label']])
print(f"\nTotal rows: {len(layer_outputs_dataframe)}")

# --- 5. Clean Up ---
lens.remove_hooks()
print("\n✨ Hooks removed successfully.")


--- Running Model to Collect Activations ---
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 66])
--- Model Run Complete ---

--- Processing Layers (mod 5 + Final) ---

--- Layer 0 ---
 recount're/dist rightfullynessships somew bothered.primarylooksminimumEarlier/fromest][:teenthth chance plain owngome/offl/or/rem somew/her/pro бы enough/de virtueeteenth embarkth-calledhere/her-seek...

--- Layer 5 ---
например рег/notsuming �/cal рег narrowed copeolder kre extensively/me moneyOccteenth different chance interest own levenest/fromcho/or bills �/heest бы-ending/de virtueeteenth{nameNature-calledadays/...

--- Layer 10 ---
напримернапример/non begrdigEntered’ex soaked imprses multit/rem catastned.googleteenthkad of interest ownaver passive/off LP/or billsFiled/he/pro/is unb/de了一eightteenth百分 Monte-zAabouts/her inc/oryst...

--- Layer 15 ---
например즉 guarCOVID一步一步CLS).__ pubb.${;</Pinterest了一是一了一了一FiledFiled of interestSELFennee了一 manoe/or purposes즉즉

In [234]:
from IPython.display import HTML, display
from html import escape

try:
    from html2image import Html2Image
except ImportError:
    Html2Image = None
    print("⚠️ html2image not installed. Install with: pip install html2image")

layer_color_map = {
    0: "#7f2704",
    5: "#b63b02",
    10: "#e95d0d",
    15: "#fd8d3c",
    20: "#fdb97d",
    25: "#fddfc0",
    "output": "#fff5eb",
}

def _color_for_layer(layer_number):
    if layer_number == "actual_output":
        return layer_color_map["output"], "black"
    numeric_layer = int(layer_number)
    color = layer_color_map.get(numeric_layer, "#cccccc")
    text_color = "white" if numeric_layer <= 10 else "black"
    return color, text_color

def _format_text_block(text, max_chars=400):
    snippet = text if len(text) <= max_chars else f"…{text[-max_chars:]}"
    return escape(snippet)

def generate_layer_visualization_html(df, prompt_text):
    legend_items = []
    columns = []

    for record in df.itertuples(index=False):
        layer_number = record.layer_number
        layer_label = record.layer_label
        generated_text = record.generated_text

        bg_color, text_color = _color_for_layer(layer_number)
        formatted_text = _format_text_block(generated_text)

        legend_items.append(
            f"""
            <div style='display:flex;align-items:center;gap:8px;'>
                <div style='width:25px;height:15px;background-color:{bg_color};border:1px solid #333;border-radius:3px;'></div>
                <span style='font-size:14px;color:#000;font-weight:500;'>{layer_label}</span>
            </div>
            """
        )

        # CHANGED: Padding reduced to 8px, font-size increased to 15px
        columns.append(
            f"""
            <div style='width:100%; box-sizing: border-box; padding:8px; background-color:{bg_color};
                        border:3px solid #2c3e50; border-radius:5px; position:relative;'>
                <div style='font-size:16px; font-weight:bold; color:{text_color}; margin-bottom:4px;'>{layer_label}</div>
                <div style='font-family:monospace; font-size:15px; color:{text_color}; white-space:pre-wrap; word-wrap:break-word; line-height:1.3;'>
{formatted_text}
                </div>
            </div>
            """
        )

    legend_html = "".join(legend_items)
    columns_html = "".join(columns)

    # CHANGED: Gap reduced to 6px for tighter stacking
    return f"""
    <div style='font-family:Arial,sans-serif;margin:20px;padding:20px;background-color:#f9f9f9;border-radius:10px;'>
        <h2 style='text-align:center;color:#2c3e50;margin-bottom:20px;font-size:32px;font-weight:bold;'>Token Across Hidden Layers</h2>
        <div style='margin:10px 0;padding:10px;background-color:#fff3cd;border-left:4px solid #ff8c00;border-radius:5px;'>
            <strong style='color:#000;'>Prompt:</strong>
            <span style='color:#000;'>{escape(prompt_text)}</span>
        </div>
        <div style='display:flex;justify-content:center;gap:15px;margin:15px 0;padding:10px;background-color:#fff;border-radius:5px;flex-wrap:wrap;border:1px solid #ddd;'>
            {legend_html}
        </div>
        <div style='display:flex; flex-direction:column; gap:6px; padding:10px 0;'>
            {columns_html}
        </div>
        <div style='margin-top:10px;padding:10px;background-color:#e8f4fd;border-radius:5px;font-size:14px;text-align:center;color:#2c3e50;font-weight:500;'>
            Unembedding deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B using hooks at every 5th hidden layer. We plot the last 400 characters of each layer's decoded text.
        </div>
    </div>
    """

layer_visualization_html = generate_layer_visualization_html(layer_outputs_dataframe, prompt)
html_document = f"<!DOCTYPE html><html lang='en'><head><meta charset='UTF-8'><title>Token Across Hidden Layers</title></head><body style='background-color:white;'>{layer_visualization_html}</body></html>"

html_output_path = "layer_visualization_html.html"
with open(html_output_path, "w", encoding="utf-8") as f:
    f.write(html_document)
print(f"✅ HTML saved to: {html_output_path}")

display(HTML(layer_visualization_html))

png_output_path = "layer_visualization_html.png"
if Html2Image is not None:
    hti = Html2Image(output_path="./")
    try:
        # Height kept large to ensure full capture
        hti.screenshot(html_str=html_document, save_as=png_output_path, size=(1200, 2500))
        print(f"✅ PNG saved to: {png_output_path}")
    except Exception as exc:
        print(f"⚠️ Could not save PNG: {exc}")
else:
    print("⚠️ Skipping PNG export because html2image is unavailable.")

✅ HTML saved to: layer_visualization_html.html


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
[71259:2898913:1121/154758.291676:ERROR:chrome/browser/chrome_browser_main.cc:1027] The use of Rosetta to run the x64 version of Chromium on Arm is neither tested nor maintained, and unexpected behavior will likely result. Please check that all tools that spawn Chromium are Arm-native.


✅ PNG saved to: layer_visualization_html.png


338918 bytes written to file /Users/mariiakoroliuk/src/externalization/externalization/plots_and_visualization/layer_visualization_html.png


In [229]:
from IPython.display import HTML, display
from html import escape

try:
    from html2image import Html2Image
except ImportError:
    Html2Image = None
    print("⚠️ html2image not installed. Install with: pip install html2image")

# --- CONFIGURATION FOR QUALITY ---
SCALE_FACTOR = 3  # 1 = Standard, 2 = High, 3 = Ultra Sharp (Retina)
BASE_WIDTH = 1200
BASE_HEIGHT = 2500
# ---------------------------------

layer_color_map = {
    0: "#7f2704",
    5: "#b63b02",
    10: "#e95d0d",
    15: "#fd8d3c",
    20: "#fdb97d",
    25: "#fddfc0",
    "output": "#fff5eb",
}

def _color_for_layer(layer_number):
    if layer_number == "actual_output":
        return layer_color_map["output"], "black"
    numeric_layer = int(layer_number)
    color = layer_color_map.get(numeric_layer, "#cccccc")
    text_color = "white" if numeric_layer <= 10 else "black"
    return color, text_color

def _format_text_block(text, max_chars=400):
    snippet = text if len(text) <= max_chars else f"…{text[-max_chars:]}"
    return escape(snippet)

def generate_layer_visualization_html(df, prompt_text):
    legend_items = []
    columns = []

    for record in df.itertuples(index=False):
        layer_number = record.layer_number
        layer_label = record.layer_label
        generated_text = record.generated_text

        bg_color, text_color = _color_for_layer(layer_number)
        formatted_text = _format_text_block(generated_text)

        legend_items.append(
            f"""
            <div style='display:flex;align-items:center;gap:8px;'>
                <div style='width:25px;height:15px;background-color:{bg_color};border:1px solid #333;border-radius:3px;'></div>
                <span style='font-size:14px;color:#000;font-weight:500;'>{layer_label}</span>
            </div>
            """
        )

        columns.append(
            f"""
            <div style='width:100%; box-sizing: border-box; padding:8px; background-color:{bg_color};
                        border:3px solid #2c3e50; border-radius:5px; position:relative;'>
                <div style='font-size:16px; font-weight:bold; color:{text_color}; margin-bottom:4px;'>{layer_label}</div>
                <div style='font-family:monospace; font-size:15px; color:{text_color}; white-space:pre-wrap; word-wrap:break-word; line-height:1.3;'>
{formatted_text}
                </div>
            </div>
            """
        )

    legend_html = "".join(legend_items)
    columns_html = "".join(columns)

    # Note: Added width: 1000px explicitly to the container.
    # This stops the text from becoming super long lines when we scale up the image.
    # It forces the text to wrap exactly how it looks on a normal screen, just higher resolution.
    return f"""
    <div style='width: 1000px; font-family:Arial,sans-serif; margin:20px; padding:20px; background-color:#f9f9f9; border-radius:10px;'>
        <h2 style='text-align:center; color:#2c3e50; margin-bottom:20px; font-size:32px; font-weight:bold;'>Token Across Hidden Layers</h2>
        <div style='margin:10px 0; padding:10px; background-color:#fff3cd; border-left:4px solid #ff8c00; border-radius:5px;'>
            <strong style='color:#000;'>Prompt:</strong>
            <span style='color:#000;'>{escape(prompt_text)}</span>
        </div>
        <div style='display:flex; justify-content:center; gap:15px; margin:15px 0; padding:10px; background-color:#fff; border-radius:5px; flex-wrap:wrap; border:1px solid #ddd;'>
            {legend_html}
        </div>
        <div style='display:flex; flex-direction:column; gap:6px; padding:10px 0;'>
            {columns_html}
        </div>
        <div style='margin-top:10px; padding:10px; background-color:#e8f4fd; border-radius:5px; font-size:14px; text-align:center; color:#2c3e50; font-weight:500;'>
            Unembedding deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B using hooks at every 5th hidden layer.
        </div>
    </div>
    """

layer_visualization_html = generate_layer_visualization_html(layer_outputs_dataframe, prompt)

# CHANGED: Applied CSS zoom to the body to scale up rendering
html_document = f"""
<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <title>Token Across Hidden Layers</title>
</head>
<body style='background-color:white; zoom: {SCALE_FACTOR};'>
    {layer_visualization_html}
</body>
</html>
"""

html_output_path = "layer_visualization_html.html"
with open(html_output_path, "w", encoding="utf-8") as f:
    f.write(html_document)
print(f"✅ HTML saved to: {html_output_path}")

# Displaying a scaled-down version for the notebook so it isn't huge on screen
display(HTML(f"<div style='transform:scale(0.8); transform-origin:top left;'>{layer_visualization_html}</div>"))

png_output_path = "layer_visualization_high_res.png"

if Html2Image is not None:
    hti = Html2Image(output_path="./")
    try:
        # CHANGED: Multiply dimensions by SCALE_FACTOR to capture the zoomed content
        capture_width = BASE_WIDTH * SCALE_FACTOR
        capture_height = BASE_HEIGHT * SCALE_FACTOR

        print(f"📸 Taking screenshot at resolution: {capture_width}x{capture_height}...")
        hti.screenshot(html_str=html_document, save_as=png_output_path, size=(capture_width, capture_height))
        print(f"✅ High-Res PNG saved to: {png_output_path}")
    except Exception as exc:
        print(f"⚠️ Could not save PNG: {exc}")
else:
    print("⚠️ Skipping PNG export because html2image is unavailable.")

✅ HTML saved to: layer_visualization_html.html


📸 Taking screenshot at resolution: 3600x7500...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
[71017:2889670:1121/153804.069993:ERROR:chrome/browser/chrome_browser_main.cc:1027] The use of Rosetta to run the x64 version of Chromium on Arm is neither tested nor maintained, and unexpected behavior will likely result. Please check that all tools that spawn Chromium are Arm-native.


✅ High-Res PNG saved to: layer_visualization_high_res.png


1123610 bytes written to file /Users/mariiakoroliuk/src/externalization/externalization/plots_and_visualization/layer_visualization_high_res.png


In [211]:
layer_outputs_dataframe

from pathlib import Path
output_path = Path("/Users/mariiakoroliuk/src/externalization/early_exit_teacher/visualizations/layer_outputs_dataframe.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

layer_outputs_dataframe.to_csv(output_path, index=False)

print(f"Layer outputs saved to {output_path.resolve()}")

Layer outputs saved to /Users/mariiakoroliuk/src/externalization/early_exit_teacher/visualizations/layer_outputs_dataframe.csv


In [216]:
def create_layer_visualization(dataframe, prompt, title="Tokens Across Hidden Layers", save_html="layer_visualization_html.html", save_png="layer_visualization_html.png", tokenizer=None):
    import html
    import matplotlib.colors as mcolors
    import matplotlib.pyplot as plt
    import pandas as pd
    import os

    # Work on a copy to avoid modifying the original
    df = dataframe.copy()

    # Try to get tokenizer if not provided
    if tokenizer is None:
        if 'tokenizer' in globals():
            tokenizer = globals()['tokenizer']

    # Define colors based on user requirements
    layer_colors = {
        15: "#F3701B",
        20: "#FDA761",
        25: "#FDD8B3",
        27: "#FFF5EB",  # Assuming 27 is final
    }
    final_layer_color = "#FFF5EB"
    default_color = "#FFFFFF"
    
    # Robustly handle layer_number sorting
    if 'layer_number' in df.columns:
        df['layer_number'] = pd.to_numeric(df['layer_number'], errors='coerce')
        df = df.sort_values('layer_number', na_position='last')
    else:
        def extract_layer_num(label):
            try:
                return int(''.join(filter(str.isdigit, str(label))))
            except:
                return 999
        df['temp_sort'] = df['layer_label'].apply(extract_layer_num)
        df = df.sort_values('temp_sort')
        if 'temp_sort' in df.columns:
            del df['temp_sort']
    
    # Prepare data structure
    layer_data_list = []
    max_tokens = 0
    
    for i, row in df.iterrows():
        layer_label = row['layer_label']
        layer_num = 0
        if 'layer_number' in row and pd.notna(row['layer_number']):
            layer_num = int(row['layer_number'])
        else:
            try:
                layer_num = int(''.join(filter(str.isdigit, str(layer_label))))
            except:
                layer_num = 0
            
        text = row['generated_text']
        if not isinstance(text, str):
            text = str(text) if text is not None else ""
        
        # Tokenize
        tokens = []
        if tokenizer:
            try:
                token_ids = tokenizer.encode(text, add_special_tokens=False)
                tokens = [tokenizer.decode([tid]) for tid in token_ids]
            except Exception as e:
                print(f"Tokenization failed for layer {layer_label}: {e}")
                tokens = text.split()
        else:
            tokens = text.split()
            
        if "Final" in str(layer_label) or layer_num == 27 or str(layer_label).lower() == "output":
             bg_color = final_layer_color
        else:
             bg_color = layer_colors.get(layer_num, default_color)
             
        layer_data_list.append({
            "label": layer_label,
            "tokens": tokens,
            "color": bg_color
        })
        max_tokens = max(max_tokens, len(tokens))

    # HTML Construction - HORIZONTAL LAYOUT
    # We want a table-like structure where rows are layers and columns are token positions.
    
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #F9F9F9; margin: 0; padding: 20px; }}
        .card {{ background-color: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); padding: 20px; overflow: auto; }}
        .header {{ margin-bottom: 20px; }}
        .title {{ font-size: 24px; font-weight: bold; color: #333; margin-bottom: 10px; }}
        .prompt-box {{ background-color: #fff3cd; border-left: 4px solid #ff8c00; padding: 10px; margin-bottom: 20px; border-radius: 4px; font-size: 14px; }}
        
        /* Grid Layout */
        .grid-container {{ 
            display: grid; 
            grid-template-columns: 100px repeat({max_tokens}, min-content); 
            gap: 2px; 
            align-items: center;
        }}
        
        .row-header {{ 
            font-weight: bold; 
            font-size: 12px; 
            color: #555;
            padding: 5px;
            text-align: right;
            position: sticky;
            left: 0;
            background-color: white;
            z-index: 10;
        }}

        .token-cell {{ 
            padding: 4px 6px; 
            border-radius: 4px; 
            font-family: monospace; 
            font-size: 13px; 
            white-space: pre; 
            border: 1px solid rgba(0,0,0,0.05);
            text-align: center;
            min-width: 20px;
        }}
        .footer {{ margin-top: 20px; font-size: 12px; color: #888; text-align: center; }}
    </style>
</head>
<body>
    <div class="card">
        <div class="header">
            <div class="title">{title}</div>
            <div class="prompt-box">
                <strong>Prompt:</strong> {html.escape(prompt)}
            </div>
        </div>
        <div class="grid-container">
    """
    
    # Iterate over layers
    for layer in layer_data_list:
        # Add row header
        html_content += f'<div class="row-header">{layer["label"]}</div>'
        
        # Add tokens for this layer
        for token in layer['tokens']:
            text_color = "black"
            token_display = html.escape(token)
            html_content += f'<div class="token-cell" style="background-color: {layer["color"]}; color: {text_color};">{token_display}</div>'
        
        # Fill remaining cells if this row has fewer tokens than max
        remaining = max_tokens - len(layer['tokens'])
        for _ in range(remaining):
             html_content += '<div></div>'

    html_content += """
        </div>
        <div class="footer">
            Unembedding deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B using python hook at each hidden layer (mod 5).
        </div>
    </div>
</body>
</html>"""

    with open(save_html, 'w', encoding='utf-8') as f: f.write(html_content)
    print(f"✅ HTML visualization saved to: {os.path.abspath(save_html)}")
    
    try:
        from html2image import Html2Image
        hti = Html2Image(output_path='./')
        # Adjust width based on content
        width = min(10000, max(1400, 150 + max_tokens * 40))
        hti.screenshot(html_str=html_content, save_as=save_png, size=(width, 800))
        print(f"✅ PNG visualization saved to: {os.path.abspath(save_png)}")
    except ImportError:
        print("html2image not installed")
    except Exception as e:
        print(f"Error saving PNG: {e}")
        
    return html_content

# Example usage
if 'layer_outputs_dataframe' in globals():
    html_viz = create_layer_visualization(
        layer_outputs_dataframe, 
        "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
        tokenizer=tokenizer if 'tokenizer' in globals() else None
    )
    from IPython.display import display, HTML
    display(HTML(html_viz))


TypeError: '<' not supported between instances of 'str' and 'int'

In [123]:
## second plot!! 

import torch
import sys
sys.path.append("../")
from shared_utils.load import get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text
from early_exit.util import get_model, load_model, load_model_from_wandb
from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode
# from early_exit_teacher.visualization import create_html_visualization, visualize_tokens_by_exit_layer, safe_decode_tokens

model_path = "../models/early_exit_20250908_layers_5_big"
artifact_path = "vkarthik095-university-of-amsterdam/early-exit/early_exit_20250908_layers_5_big:v0"
base_model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
config_path = "../config_deepseek.yaml"
device = "cpu" 

In [124]:
tokenizer = get_tokenizer(base_model)
config = configs_from_yaml(config_path, tokenizer.eos_token_id)

base_model = get_model(base_model, config['model'], device)
model = replace_attention_layers(base_model, config['lora'], device)
model = load_model_from_wandb(model, model_path, artifact_path)

print(f"Model loaded w exitable layers: {model.exitable_layer_idxs}")

replacing generate_layer_type_without_early_exit_decision_head layer model.layers.0
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.1
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.2
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.3
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.4
replacing layer model.layers.5
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.6
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.7
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.8
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.9
replacing layer model.layers.10
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.11
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.12
replacing g

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: maria-koroliuk (vkarthik095-university-of-amsterdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Downloading large artifact early_exit_20250908_layers_5_big:v0, 1887.29MB. 5 files... 
wandb:   5 of 5 files downloaded.  
Done. 0:0:2.1 (893.2MB/s)


Model loaded w exitable layers: tensor([ 5., 10., 15., 20., 25., inf])


In [169]:
set_transformer_early_exit_mode(model, 'free_generate')

prompt = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
system_prompt = "You are a helpful assistant."
prefiller = ""

config['generation']['max_new_tokens'] = 400
with torch.no_grad():
    try:
        free_generate_response, exit_info = generate_text(
            model=model,
            prompt=prompt,
            system_prompt=system_prompt,
            prefiller=prefiller,
            tokenizer=tokenizer,
            generation_config=config['generation'],
            device=device
        )
        
        print(f"Free Generate Response: {free_generate_response,}")
        print(f"Exit info: {exit_info}")
        
    except Exception as e:
        print(f"Free generate mode failed: {e}")

full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 50])
Free Generate Response: ("<｜begin▁of▁sentence｜>You are a helpful assistant.<｜User｜>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<｜Assistant｜><think>\nFirst, I need to determine how many clips Natalia sold in April. According to the problem, she sold 48 clips to her friends.\n\nNext, I'll calculate the number of clips she sold in5February. The problem states that she sold half as many clips in May as in April. Since she sold 48 clips in April, half of that would be 24 clips in May.\n\nFinally, to find out how many clips Natalia sold altogether in April and May, I'll add the number of clips sold by her friends on each occasion. So, 48 clips in April plus 24 clips in5February equals 72 clips total.\n</think>\n\n**Solution:**\n\n1. **Clips Sold in April:**\n   \n   Natalia sold 48 clips to he

In [219]:

def visualize_tokens_by_exit_layer(token_strings, exit_layers, early_exit_layer_idxs=None, 
                                  title="Token Early Exit Visualization", prompt="", save_html=None):
    """
    Visualize tokens colored by their early exit layers in Jupyter notebook or save as HTML.
    
    Args:
        token_strings: List of token strings
        exit_layers: List of exit layer indices (same length as token_strings)
        early_exit_layer_idxs: List/tensor of available early exit layers (optional)
        title: Title for the visualization
        prompt: Prompt text to display at the top (optional)
        save_html: Path to save HTML file (optional). If provided, saves to file instead of returning HTML object.
    
    Returns:
        IPython.display.HTML object for rendering in notebook (if save_html is None)
        or None (if save_html is provided)
    """
    
    # Get all unique layers and create color mapping
    unique_layers = sorted(set(exit_layers))
    if early_exit_layer_idxs is not None:
        # Include all possible layers even if not used
        all_layers = list(early_exit_layer_idxs) + [27]  # 27 for final layer
        unique_layers = sorted(set(all_layers))
    
    # Create color mapping using Oranges_r colormap (darker orange = lower/earlier layers)
    cmap = plt.colormaps.get_cmap('Oranges_r')
    norm = mcolors.Normalize(vmin=0, vmax=len(unique_layers)-1)
    
    layer_colors = {}
    for i, layer in enumerate(unique_layers):
        color = cmap(norm(i))
        # Convert to hex color
        hex_color = '#{:02x}{:02x}{:02x}'.format(
            int(color[0] * 255),
            int(color[1] * 255),
            int(color[2] * 255)
        )
        layer_colors[layer] = hex_color
    
    # Start building HTML
    html_content = f"""
    <div style="font-family: Arial, sans-serif; margin: 20px; padding: 20px; 
                background-color: #f9f9f9; border-radius: 10px;">
        <h3 style="text-align: center; color: #333; margin-bottom: 20px;">{title}</h3>
    """
    
    # Add prompt if provided
    if prompt:
        html_content += f"""
        <!-- Prompt -->
        <div style="margin: 15px 0; padding: 12px; background-color: #fff3cd; 
                    border-left: 4px solid #ff8c00; border-radius: 5px;">
            <strong style="color: #000;">Prompt:</strong> 
            <span style="color: #000;">{html.escape(prompt)}</span>
        </div>
        """
    
    html_content += """
        <!-- Legend -->
        <div style="display: flex; justify-content: center; gap: 15px; 
                    margin: 20px 0; padding: 15px; background-color: #fff; 
                    border-radius: 5px; flex-wrap: wrap; border: 1px solid #ddd;">
    """
    
    # Add legend items
    for layer in unique_layers:
        if layer in [l for l in exit_layers]:  # Only show layers that are actually used
            layer_name = f"Layer {layer}" if layer != 27 else "Final Layer"
            color = layer_colors[layer]
            # Determine text color based on background brightness
            r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
            brightness = (r * 299 + g * 587 + b * 114) / 1000
            text_color = "white" if brightness < 128 else "black"
            
            html_content += f"""
                <div style="display: flex; align-items: center; gap: 8px;">
                    <div style="width: 25px; height: 15px; background-color: {color}; 
                                border: 1px solid #333; border-radius: 3px;"></div>
                    <span style="font-size: 14px; color: #000; font-weight: 500;">{layer_name}</span>
                </div>
            """
    
    html_content += """
        </div>
        
        <!-- Tokens -->
        <div style="line-height: 2.5; word-wrap: break-word; padding: 15px; 
                    background-color: #fff; border-radius: 5px; border: 1px solid #ddd;">
    """
    
    # Add tokens
    for token, exit_layer in zip(token_strings, exit_layers):
        color = layer_colors[exit_layer]
    # Add tokens
    for token, exit_layer in zip(token_strings, exit_layers):
        color = layer_colors[exit_layer]
        # Escape special characters and handle unicode properly
        token_display = html.escape(token, quote=False)
        # Replace common whitespace and control characters with visible representations
        token_display = token_display.replace('\n', '\\n').replace('\t', '\\t').replace('\r', '\\r')
        # Handle other special characters
        token_display = token_display.replace('\u00a0', '[NBSP]')  # Non-breaking space
        token_display = token_display.replace('\ufeff', '[BOM]')   # Byte order mark
        # Replace any remaining non-printable characters
        token_display = ''.join(char if char.isprintable() or char in ' \n\t' else f'[U+{ord(char):04X}]' for char in token_display)
        
        # Determine text color based on background brightness
        r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
        brightness = (r * 299 + g * 587 + b * 114) / 1000
        text_color = "white" if brightness < 128 else "black"
        
        html_content += f"""<span style="display: inline-block; padding: 4px 8px; margin: 2px; 
                                      border-radius: 4px; border: 1px solid #666; 
                                      font-family: monospace; font-size: 13px; 
                                      background-color: {color}; color: {text_color}; 
                                      font-weight: bold; max-width: 200px; 
                                      overflow-wrap: break-word; vertical-align: middle;">{token_display}</span>"""
    
    html_content += """
        </div>
        
        <!-- Statistics -->
        <div style="margin-top: 15px; padding: 10px; background-color: #e8f4fd; 
                    border-radius: 5px; font-family: monospace; font-size: 13px;">
    """
    
    # Add statistics
    layer_counts = {}
    for layer in unique_layers:
        count = exit_layers.count(layer)
        if count > 0:  # Only show layers that are used
            layer_counts[layer] = count
    
    stats_text = f"Total tokens: {len(token_strings)} | "
    for layer, count in layer_counts.items():
        percentage = (count / len(exit_layers) * 100) if len(exit_layers) > 0 else 0
        layer_name = f"Layer {layer}" if layer != 27 else "Final"
        stats_text += f"{layer_name}: {count} ({percentage:.1f}%) | "
    
    html_content += stats_text.rstrip(' |')
    
    html_content += """
        </div>
    </div>
    """
    
    if save_html:
        # Create complete HTML document for standalone file
        full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
</head>
<body>
{html_content}
</body>
</html>"""
        
        # Save to file
        with open(save_html, 'w', encoding='utf-8') as f:
            f.write(full_html)
        print(f"HTML visualization saved to: {save_html}")
        return html_content
    else:
        return html_content
    


In [ ]:

def safe_decode_tokens(tokenizer, token_ids):
    """
    Safely decode tokens: only keep alphanumeric and punctuation characters.
    Remove special characters like newlines, tabs, and EOS tokens.
    Everything else shows token ID.
    """
    tokens = []
    for tid in token_ids:
        # Skip EOS/BOS tokens (common token IDs for these)
        if tid in [tokenizer.eos_token_id, tokenizer.bos_token_id, tokenizer.pad_token_id]:
            continue
            
        # Try to decode the token
        tok = tokenizer.decode([tid], skip_special_tokens=False)
        
        # Remove control characters and special whitespace
        tok = tok.replace('\n', '').replace('\r', '').replace('\t', '')
        
        # Skip empty tokens after cleaning
        if not tok or tok.isspace():
            continue
        
        # Check if all characters are alphanumeric, space, or punctuation
        if tok and all(c.isalnum() or c == ' ' or c in string.punctuation for c in tok):
            tokens.append(tok)
        else:
            # Skip rather than showing token ID for cleaner output
            continue
                
    
    return tokens



In [235]:
# tokens = [tokenizer.decode([token]) for token in exit_info[0][0, 20:]]
gen_len = exit_info[1][0].shape[-1]
tokens = safe_decode_tokens(tokenizer, exit_info[0][0, -gen_len:])
layers = [27 if item == torch.inf or item == -1 else int(item) for item in exit_info[1][0]]

# Display the visualization
from IPython.display import HTML
html_viz = visualize_tokens_by_exit_layer(
    tokens, 
    layers, 
    [int(item) for item in model.exitable_layer_idxs[:-1]], 
    title="Proof of Concept: Early Exit Mechanism Successfully Engages",
    prompt=prompt
)
display(HTML(html_viz))

# Save the visualization as PNG
try:
    from html2image import Html2Image
    hti = Html2Image(output_path='./')
    
    # Create full HTML document for better rendering
    full_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ margin: 0; padding: 20px; background-color: white; }}
        </style>
    </head>
    <body>
        {html_viz}
    </body>
    </html>
    """
    
    # Save as PNG
    output_file = 'token_exit_visualization_committed.png'
    hti.screenshot(html_str=full_html, save_as=output_file, size=(1400, 1000))
    print(f"✅ Visualization saved as: {output_file}")
    
except ImportError:
    print("⚠️ html2image not installed. Install with: pip install html2image")
    print("   Note: Also requires Chrome/Chromium browser to be installed.")
except Exception as e:
    print(f"⚠️ Could not save PNG: {e}")
    print("   Saving as HTML instead...")
    # Fallback: save as HTML file
    html_file = 'token_exit_visualization_committed.html'
    with open(html_file, 'w', encoding='utf-8') as f:
        f.write(f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Token Exit Visualization</title>
</head>
<body>
{html_viz}
</body>
</html>""")
    print(f"✅ Saved as HTML: {html_file} (Open in browser to view)")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
[71357:2902412:1121/155116.675448:ERROR:chrome/browser/chrome_browser_main.cc:1027] The use of Rosetta to run the x64 version of Chromium on Arm is neither tested nor maintained, and unexpected behavior will likely result. Please check that all tools that spawn Chromium are Arm-native.


✅ Visualization saved as: token_exit_visualization_committed.png


177278 bytes written to file /Users/mariiakoroliuk/src/externalization/externalization/plots_and_visualization/token_exit_visualization_committed.png


In [183]:
!pip install html2image

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [html2image]


In [222]:
import pandas as pd
import html
from IPython.display import HTML, display

def visualize_logit_lens(df, prompt_text="", title="tokens across hidden layers"):
    """
    Visualizes Logit Lens data from a DataFrame with specific styling.
    
    Args:
        df: DataFrame with 'layer_number' and either 'tokens' (list) or 'generated_text' (string).
        prompt_text: The prompt to display at the top.
        title: Title of the visualization.
    """
    
    # --- Configuration ---
    bg_color = "#F9F9F9"
    
    # Specific row colors (applied to token cards)
    layer_colors = {
        15: "#F3701B",
        20: "#FDA761",
        25: "#FDD8B3",
    }
    final_layer_color = "#FFF5EB"
    default_token_color = "#FFFFFF"  # Neutral for "chaos" layers (0, 5, 10)
    
    # Text colors for contrast
    text_colors = {
        15: "white",  # Dark orange needs white text
        20: "black",
        25: "black",
    }
    default_text_color = "black"

    # Find the final layer number to apply the cream color
    max_layer = df['layer_number'].max()
    
    # --- HTML Generation ---
    html_content = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px; background-color: {bg_color}; border-radius: 10px;">
        
        <!-- Title -->
        <h2 style="text-align: center; color: #333; margin-bottom: 20px; text-transform: capitalize;">{title}</h2>
        
        <!-- Prompt Section -->
        <div style="margin-bottom: 30px; padding: 15px; background-color: #fff; border-left: 5px solid #F3701B; border-radius: 5px; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
            <strong style="display:block; margin-bottom: 8px; color: #555;">Prompt:</strong>
            <div style="font-family: monospace; font-size: 13px; color: #333; line-height: 1.5;">{html.escape(prompt_text)}</div>
        </div>

        <!-- Visualization Container (Horizontal Scroll) -->
        <div style="overflow-x: auto; padding-bottom: 10px;">
            <div style="display: table; border-collapse: separate; border-spacing: 0 10px;">
    """

    # Iterate through layers (Rows)
    # Ensure DataFrame is sorted by layer number
    df_sorted = df.sort_values('layer_number')
    
    for _, row in df_sorted.iterrows():
        layer_num = row['layer_number']
        
        # Determine content (handle list of tokens or raw text)
        if 'tokens' in row and isinstance(row['tokens'], list):
            tokens = row['tokens']
        elif 'generated_text' in row:
             # Fallback: split by space if explicit tokens aren't there
            tokens = str(row['generated_text']).split()
        else:
            tokens = ["(No data)"]

        # Determine styling for this layer
        is_final = (layer_num == max_layer) or (str(layer_num).lower() == 'output')
        
        if is_final:
            card_bg = final_layer_color
            card_text = default_text_color
            label_text = "Output"
        elif layer_num in layer_colors:
            card_bg = layer_colors[layer_num]
            card_text = text_colors.get(layer_num, default_text_color)
            label_text = f"Layer {layer_num}"
        else:
            card_bg = default_token_color
            card_text = default_text_color
            label_text = f"Layer {layer_num}"

        # Row Start
        html_content += f"""
                <div style="display: table-row;">
                    <!-- Layer Label (Sticky Left) -->
                    <div style="display: table-cell; vertical-align: top; padding-right: 15px; white-space: nowrap; font-weight: bold; color: #444; position: sticky; left: 0; background-color: {bg_color}; z-index: 10;">
                        {label_text}
                    </div>
                    
                    <!-- Token Sequence -->
                    <div style="display: table-cell; white-space: nowrap;">
        """
        
        # Render Tokens
        for token in tokens:
            safe_token = html.escape(str(token))
            # Handle spaces for visualization
            if safe_token.strip() == "":
                safe_token = "&nbsp;"
            
            html_content += f"""
                        <span style="
                            display: inline-block;
                            padding: 4px 8px;
                            margin: 2px;
                            border-radius: 4px;
                            border: 1px solid #ddd;
                            font-family: monospace;
                            font-size: 13px;
                            background-color: {card_bg};
                            color: {card_text};
                            font-weight: bold;
                            min-width: 10px;
                            text-align: center;
                        ">{safe_token}</span>
            """
            
        html_content += """
                    </div>
                </div>
        """

    html_content += """
            </div>
        </div>
    </div>
    """
    
    display(HTML(html_content))

# --- Execute Visualization ---
# Assuming layer_outputs_dataframe is in memory
prompt = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"

visualize_logit_lens(
    layer_outputs_dataframe, 
    prompt_text=prompt,
    title="Tokens Across Hidden Layers"
)

TypeError: '>=' not supported between instances of 'int' and 'str'